In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the data
raw = pd.read_csv('gcdg_zaf.csv', low_memory=False)
print(raw.shape)
print(raw['subjid'].nunique(), 'children')

(4176, 363)
2389 children


In [3]:
print(raw['agedays'].describe())
print('missing age:', raw['agedays'].isna().sum())

count    4175.000000
mean      943.178683
std       450.780117
min       125.000000
25%       721.000000
50%       782.000000
75%      1461.000000
max      1461.000000
Name: agedays, dtype: float64
missing age: 1


In [4]:
df = raw[raw['agedays'] <= 3 * 365].copy()
df['age_months'] = df['agedays'] / 30.4375
print('rows:', len(df))
print('children:', df['subjid'].nunique())
print(df['age_months'].min(), df['age_months'].max())

rows: 2561
children: 2033
4.1067761806981515 29.305954825462013


In [5]:
# Checking Data Quality
meta_cols = ['ctrycd', 'cohort', 'cohortn', 'subjid', 'agedays', 'sex', 'gagebrth']
item_cols = [col for col in df.columns if col not in meta_cols + ['age_months']]
print(len(item_cols), 'item columns')
print(df[meta_cols].isna().sum())

356 item columns
ctrycd       0
cohort       0
cohortn      0
subjid       0
agedays      0
sex          0
gagebrth    53
dtype: int64


In [6]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate child + age:', df.duplicated(['subjid', 'agedays']).sum())

duplicate rows: 0
duplicate child + age: 0


In [7]:
values = pd.unique(df[item_cols].values.ravel())
print(sorted(v for v in values if v == v))

[np.float64(0.0), np.float64(1.0)]


In [8]:
print(df['sex'].value_counts())
print('children with more than one sex:', (df.groupby('subjid')['sex'].nunique() > 1).sum())
print('children with more than one gestational age:', (df.groupby('subjid')['gagebrth'].nunique() > 1).sum())

sex
Female    1312
Male      1249
Name: count, dtype: int64
children with more than one sex: 0
children with more than one gestational age: 0


In [9]:
answered = df[item_cols].notna().sum(axis=1)
print(answered.describe())
print('rows with no items answered:', (answered == 0).sum())

count    2561.000000
mean       72.866458
std        69.354925
min         8.000000
25%        28.000000
50%        29.000000
75%       166.000000
max       201.000000
dtype: float64
rows with no items answered: 0


In [10]:
never_answered = [col for col in item_cols if df[col].isna().all()]
print(len(never_answered), 'items never answered:', never_answered)

21 items never answered: ['vinxxc031', 'vinxxc032', 'vinxxc033', 'vinxxc034', 'vinxxc035', 'vinxxc036', 'vinxxc037', 'vinxxc038', 'vinxxc039', 'vinxxc040', 'vinxxc041', 'vinxxc042', 'vinxxc043', 'vinxxc044', 'vinxxc045', 'vinxxc046', 'vinxxc047', 'vinxxc048', 'vinxxc049', 'vinxxc050', 'vinxxc051']


In [11]:
vineland = [col for col in item_cols if col.startswith('vin')]
others = [col for col in item_cols if not col.startswith('vin')]
has_vineland = df[vineland].notna().any(axis=1)
has_others = df[others].notna().any(axis=1)
print('Vineland only:', (has_vineland & ~has_others).sum())
print('other instruments only:', (~has_vineland & has_others).sum())
print('both:', (has_vineland & has_others).sum())

Vineland only: 1801
other instruments only: 760
both: 0


In [12]:
print(df.groupby('subjid').size().value_counts())
print(pd.cut(df['age_months'], [0, 6, 12, 18, 24, 30]).value_counts().sort_index())

1    1551
2     436
3      46
Name: count, dtype: int64
age_months
(0, 6]       216
(6, 12]      274
(12, 18]     270
(18, 24]     526
(24, 30]    1275
Name: count, dtype: int64


In [13]:
item_cols = [col for col in item_cols if col not in never_answered]
print(len(item_cols), 'items kept')

335 items kept


In [14]:
import json

labels = json.load(open('item_labels.json'))
print(len(labels), 'labels')
print('items without a label:', [col for col in item_cols if col not in labels])

335 labels
items without a label: []


In [15]:
prefix_domain = {
    'by1pdd': 'motor',
    'by1mdd': 'cognitive',
    'gricgd': 'cognitive',
    'griehd': 'motor',
    'grigmd': 'motor',
    'grihsd': 'language',
    'sgrehd': 'motor',
    'sgrfmd': 'motor',
    'sgrgmd': 'motor',
    'sgrhsd': 'language',
    'sgrred': 'social_emotional',
    'vinxxc': 'social_emotional',
}

In [16]:
overrides = {
    'by1mdd': {
        'motor': [37, 44, 46, 49, 51, 54, 56, 60, 64, 69, 70, 77, 82, 87, 98, 100, 103, 105, 107, 108, 109, 111, 112, 114, 118, 119, 123, 125],
        'language': [47, 48, 55, 79, 84, 85, 89, 90, 94, 101, 106, 113, 116, 124],
        'social_emotional': [42, 53, 58, 61, 65, 76, 81, 97, 117],
    },
    'gricgd': {
        'motor': [4, 7, 8, 11, 12, 13, 23],
    },
    'griehd': {
        'cognitive': [5, 6, 7, 8, 13, 22, 208, 210],
    },
    'sgrehd': {
        'cognitive': [20, 25],
    },
    'sgrfmd': {
        'cognitive': [8],
    },
    'grihsd': {
        'cognitive': [201, 210],
    },
    'sgrhsd': {
        'social_emotional': [10],
        'cognitive': [15],
    },
    'sgrred': {
        'language': [6, 13, 30],
        'cognitive': [18, 29, 32, 37],
    },
    'vinxxc': {
        'motor': [4, 8, 11, 14, 15, 25],
        'cognitive': [5, 12, 22],
        'language': [6, 13, 16, 24],
    },
}

In [17]:
item_map = {}
for item in item_cols:
    item_map[item] = prefix_domain[item[:6]]

for prefix, groups in overrides.items():
    for domain, numbers in groups.items():
        for number in numbers:
            item_map[prefix + str(number).zfill(3)] = domain

map_table = pd.DataFrame({'item': item_cols})
map_table['prefix'] = map_table['item'].str[:6]
map_table['domain'] = map_table['item'].map(item_map)
map_table['label'] = map_table['item'].map(labels)
print(map_table['domain'].value_counts())

domain
motor               146
cognitive            82
social_emotional     54
language             53
Name: count, dtype: int64


In [18]:
print(pd.crosstab(map_table['prefix'], map_table['domain']))

domain  cognitive  language  motor  social_emotional
prefix                                              
by1mdd         37        14     28                 9
by1pdd          0         0     39                 0
gricgd         24         0      7                 0
griehd          8         0     22                 0
grigmd          0         0     30                 0
grihsd          2        29      0                 0
sgrehd          2         0      4                 0
sgrfmd          1         0      4                 0
sgrgmd          0         0      6                 0
sgrhsd          1         3      0                 1
sgrred          4         3      0                28
vinxxc          3         4      6                16


In [19]:
for domain in ['social_emotional', 'language', 'cognitive', 'motor']:
    print('\n' + domain.upper())
    subset = map_table[map_table['domain'] == domain]
    for _, row in subset.iterrows():
        print(row['item'], '-', row['label'])


SOCIAL_EMOTIONAL
by1mdd042 - Aware of strange situation
by1mdd053 - mirror image approach
by1mdd058 - Discriminates strangers
by1mdd061 - Likes frolic play
by1mdd065 - smiles at mirror (5.4 months)
by1mdd076 - playful response to mirror
by1mdd081 - cooperates in games (is this some other scale?)
by1mdd097 - repeats performance laughed at
by1mdd117 - shows shoes or other clothing, or own toy
sgrhsd010 - Laughs aloud
sgrred007 - Follows moving person
sgrred008 - Coos and smiles
sgrred009 - Friendly to strangers
sgrred010 - Resist ring being taken away
sgrred011 - Frolics when played with
sgrred012 - Stops crying when spoken to
sgrred014 - Anticipate being lifted
sgrred015 - Holds a spoon
sgrred016 - Stretches to be taken
sgrred017 - Drinks from a cup
sgrred019 - Reacts to mirror image
sgrred020 - Knows strangers
sgrred021 - Enthusiastic
sgrred022 - Displeased when toy taken
sgrred023 - Helps hold cup
sgrred024 - Pulls off hat
sgrred025 - Smiles, plays mirror
sgrred026 - Waves bye bye
sg

In [20]:
domains = ['cognitive', 'motor', 'language', 'social_emotional']
for domain in domains:
    cols = map_table.loc[map_table['domain'] == domain, 'item']
    df['n_' + domain] = df[cols].notna().sum(axis=1)

df['family'] = np.where(has_vineland, 'vineland', 'other')
print(df.groupby('family')[['n_' + d for d in domains]].describe().T.loc[(slice(None), ['min', 'mean', 'max']), :])

family                       other   vineland
n_cognitive        min   39.000000   1.000000
n_motor            min   80.000000   1.000000
n_language         min   24.000000   0.000000
n_social_emotional min   22.000000   4.000000
n_cognitive        mean  43.700000   2.951694
n_motor            mean  83.980263   5.542476
n_language         mean  28.339474   3.980566
n_social_emotional mean  22.638158  15.749028
n_cognitive        max   52.000000   3.000000
n_motor            max   91.000000   6.000000
n_language         max   36.000000   4.000000
n_social_emotional max   23.000000  16.000000


In [21]:
json.dump(item_map, open('item_domain_map.json', 'w'), indent=1)

In [22]:
min_items = 3

for domain in domains:
    cols = map_table.loc[map_table['domain'] == domain, 'item']
    df[domain] = df[cols].mean(axis=1)
    df.loc[df['n_' + domain] < min_items, domain] = np.nan

print(df[domains].describe())

         cognitive        motor     language  social_emotional
count  2481.000000  2560.000000  2554.000000       2561.000000
mean      0.838983     0.837390     0.861393          0.848855
std       0.197167     0.171259     0.179964          0.118319
min       0.000000     0.000000     0.250000          0.125000
25%       0.666667     0.744815     0.750000          0.800000
50%       0.923077     0.835165     1.000000          0.875000
75%       1.000000     1.000000     1.000000          0.937500
max       1.000000     1.000000     1.000000          1.000000


In [23]:
print(df[domains].isna().sum())
print('rows with all four domains:', df[domains].notna().all(axis=1).sum())

cognitive           80
motor                1
language             7
social_emotional     0
dtype: int64
rows with all four domains: 2481


In [24]:
print(df.groupby('family')[domains].mean())
print((df[domains] == 1).mean())

          cognitive     motor  language  social_emotional
family                                                   
other      0.801120  0.803949  0.771818          0.864424
vineland   0.855704  0.851509  0.899340          0.842286
cognitive           0.443186
motor               0.359625
language            0.508395
social_emotional    0.117142
dtype: float64


In [25]:
# Split train and test sets by child
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['subjid']))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print('train rows:', len(train), 'children:', train['subjid'].nunique())
print('test rows:', len(test), 'children:', test['subjid'].nunique())
print('children in both:', len(set(train['subjid']) & set(test['subjid'])))

train rows: 2028 children: 1626
test rows: 533 children: 407
children in both: 0


In [26]:
# Define age bands and build the norms table
print(train['family'].value_counts(normalize=True))
print(test['family'].value_counts(normalize=True))

family
vineland    0.706114
other       0.293886
Name: proportion, dtype: float64
family
vineland    0.692308
other       0.307692
Name: proportion, dtype: float64


In [27]:
print(pd.cut(train['age_months'], range(3, 33, 3), right=False).value_counts().sort_index())

age_months
[3, 6)      174
[6, 9)      202
[9, 12)       5
[12, 15)    174
[15, 18)     41
[18, 21)      0
[21, 24)    418
[24, 27)    784
[27, 30)    230
Name: count, dtype: int64


In [28]:
bins = [3, 6, 9, 18, 21, 24, 27, 30]
train['age_band'] = pd.cut(train['age_months'], bins, right=False)
test['age_band'] = pd.cut(test['age_months'], bins, right=False)

print(train['age_band'].value_counts().sort_index())
print(test['age_band'].value_counts().sort_index())

age_band
[3, 6)      174
[6, 9)      202
[9, 18)     220
[18, 21)      0
[21, 24)    418
[24, 27)    784
[27, 30)    230
Name: count, dtype: int64
age_band
[3, 6)       42
[6, 9)       67
[9, 18)      55
[18, 21)      0
[21, 24)    108
[24, 27)    201
[27, 30)     60
Name: count, dtype: int64


In [29]:
rows = []
for band, group in train.groupby('age_band', observed=True):
    for domain in domains:
        values = group[domain].dropna()
        rows.append({
            'age_low': band.left,
            'age_high': band.right,
            'domain': domain,
            'n': len(values),
            'mean': values.mean(),
            'std': values.std(),
        })

norms = pd.DataFrame(rows)
print(norms.round(3).to_string())

    age_low  age_high            domain    n   mean    std
0         3         6         cognitive  174  0.741  0.154
1         3         6             motor  174  0.719  0.156
2         3         6          language  174  0.657  0.118
3         3         6  social_emotional  174  0.846  0.119
4         6         9         cognitive  202  0.816  0.116
5         6         9             motor  202  0.812  0.121
6         6         9          language  202  0.738  0.121
7         6         9  social_emotional  202  0.868  0.130
8         9        18         cognitive  220  0.834  0.121
9         9        18             motor  220  0.867  0.105
10        9        18          language  220  0.893  0.088
11        9        18  social_emotional  220  0.874  0.106
12       21        24         cognitive  393  0.843  0.230
13       21        24             motor  418  0.844  0.174
14       21        24          language  416  0.863  0.206
15       21        24  social_emotional  418  0.817  0.1

In [30]:
norms.to_json('norms.json', orient='records', indent=1)

In [31]:
# Convert scores to age-normalized z-scores
def add_z(data):
    data = data.copy()
    data['band_low'] = data['age_band'].map(lambda b: b.left).astype(float)
    for domain in domains:
        table = norms[norms['domain'] == domain].set_index('age_low')
        mean = data['band_low'].map(table['mean'])
        std = data['band_low'].map(table['std'])
        data['z_' + domain] = (data[domain] - mean) / std
    return data

train = add_z(train)
test = add_z(test)
z_cols = ['z_' + d for d in domains]

In [32]:
print(train[z_cols].describe().round(2))
print(test[z_cols].describe().round(2))

       z_cognitive  z_motor  z_language  z_social_emotional
count      1965.00  2028.00     2024.00             2028.00
mean         -0.00    -0.00        0.00               -0.00
std           1.00     1.00        1.00                1.00
min          -5.91    -6.04       -6.61               -7.29
25%          -0.77    -0.54       -0.55               -0.53
50%           0.49     0.30        0.54                0.32
75%           0.73     0.79        0.54                0.75
max           1.58     1.48        2.20                1.44
       z_cognitive  z_motor  z_language  z_social_emotional
count       516.00   532.00      530.00              533.00
mean          0.06    -0.04       -0.06               -0.02
std           0.92     1.00        1.02                0.97
min          -3.81    -3.62       -3.84               -5.65
25%          -0.77    -0.54       -0.67               -0.53
50%           0.49     0.11        0.36                0.32
75%           0.73     0.79        0.54 

In [33]:
print(train.groupby('age_band', observed=True)[z_cols].mean().round(2))

          z_cognitive  z_motor  z_language  z_social_emotional
age_band                                                      
[3, 6)           -0.0      0.0         0.0                 0.0
[6, 9)           -0.0      0.0         0.0                -0.0
[9, 18)          -0.0     -0.0        -0.0                -0.0
[21, 24)          0.0     -0.0         0.0                -0.0
[24, 27)          0.0     -0.0        -0.0                -0.0
[27, 30)         -0.0     -0.0        -0.0                 0.0


In [34]:
print('train below -2:')
print((train[z_cols] < -2).sum())
print('test below -2:')
print((test[z_cols] < -2).sum())

train below -2:
z_cognitive           146
z_motor                59
z_language             99
z_social_emotional     96
dtype: int64
test below -2:
z_cognitive           26
z_motor               19
z_language            30
z_social_emotional    24
dtype: int64


In [35]:
keep = ['subjid', 'agedays', 'age_months', 'age_band', 'sex', 'family'] + domains + z_cols
train_out = train[keep].copy()
test_out = test[keep].copy()
train_out['age_band'] = train_out['age_band'].astype(str)
test_out['age_band'] = test_out['age_band'].astype(str)
train_out.to_csv('train.csv', index=False)
test_out.to_csv('test.csv', index=False)
print(train_out.shape, test_out.shape)

(2028, 14) (533, 14)


In [36]:
# Handle missing domains
print('train rows with a missing z-score:', train[z_cols].isna().any(axis=1).sum())
print('test rows with a missing z-score:', test[z_cols].isna().any(axis=1).sum())
print(train[z_cols].isna().sum())
print(train.loc[train[z_cols].isna().any(axis=1), 'family'].value_counts())

train rows with a missing z-score: 63
test rows with a missing z-score: 17
z_cognitive           63
z_motor                0
z_language             4
z_social_emotional     0
dtype: int64
family
vineland    63
Name: count, dtype: int64


In [37]:
complete = train.dropna(subset=z_cols)
print(complete[z_cols].corr().round(2))

                    z_cognitive  z_motor  z_language  z_social_emotional
z_cognitive                1.00     0.46        0.29                0.38
z_motor                    0.46     1.00        0.31                0.38
z_language                 0.29     0.31        1.00                0.38
z_social_emotional         0.38     0.38        0.38                1.00


In [38]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, GroupKFold

for target in z_cols:
    others = [col for col in z_cols if col != target]
    scores = cross_val_score(LinearRegression(), complete[others], complete[target], cv=GroupKFold(5), groups=complete['subjid'], scoring='r2')
    print(target, 'predicted from the other three domains, R2:', round(scores.mean(), 2))
    if target != 'z_social_emotional':
        scores = cross_val_score(LinearRegression(), complete[['z_social_emotional']], complete[target], cv=GroupKFold(5), groups=complete['subjid'], scoring='r2')
        print(target, 'predicted from social-emotional only, R2:', round(scores.mean(), 2))

z_cognitive predicted from the other three domains, R2: 0.25
z_cognitive predicted from social-emotional only, R2: 0.13
z_motor predicted from the other three domains, R2: 0.27
z_motor predicted from social-emotional only, R2: 0.14
z_language predicted from the other three domains, R2: 0.18
z_language predicted from social-emotional only, R2: 0.14
z_social_emotional predicted from the other three domains, R2: 0.25


In [39]:
train['complete'] = train[z_cols].notna().all(axis=1)
test['complete'] = test[z_cols].notna().all(axis=1)
print('train complete:', train['complete'].sum(), 'of', len(train))
print('test complete:', test['complete'].sum(), 'of', len(test))

train complete: 1965 of 2028
test complete: 516 of 533


In [40]:
keep = ['subjid', 'agedays', 'age_months', 'age_band', 'sex', 'family'] + domains + z_cols + ['complete']
train_out = train[keep].copy()
test_out = test[keep].copy()
train_out['age_band'] = train_out['age_band'].astype(str)
test_out['age_band'] = test_out['age_band'].astype(str)
train_out.to_csv('train.csv', index=False)
test_out.to_csv('test.csv', index=False)
print(train_out.shape, test_out.shape)

(2028, 15) (533, 15)


In [41]:
# Record supported age bands and domain coverage
all_rows = pd.concat([train, test])

coverage = all_rows.groupby('age_band', observed=True).agg(
    rows=('subjid', 'size'),
    min_age=('age_months', 'min'),
    max_age=('age_months', 'max'),
    complete=('complete', 'mean'),
)
print(coverage.round(2))

          rows  min_age  max_age  complete
age_band                                  
[3, 6)     216     4.11     5.91      1.00
[6, 9)     269     6.01     8.31      1.00
[9, 18)    275    11.99    16.00      1.00
[21, 24)   526    22.51    23.98      0.94
[24, 27)   985    24.11    26.91      0.96
[27, 30)   290    27.01    29.31      0.97


In [42]:
print(pd.crosstab(all_rows['age_band'], all_rows['family']))
print(all_rows.groupby('age_band', observed=True)[['n_' + d for d in domains]].mean().round(1))

family    other  vineland
age_band                 
[3, 6)      216         0
[6, 9)      269         0
[9, 18)     275         0
[21, 24)      0       526
[24, 27)      0       985
[27, 30)      0       290
          n_cognitive  n_motor  n_language  n_social_emotional
age_band                                                      
[3, 6)           39.0     80.0        24.0                23.0
[6, 9)           39.0     80.0        24.0                23.0
[9, 18)          52.0     91.0        36.0                22.0
[21, 24)          2.9      5.6         4.0                15.8
[24, 27)          3.0      5.5         4.0                15.7
[27, 30)          3.0      5.6         4.0                15.7


In [43]:
band_items = {}
for band, group in all_rows.groupby('age_band', observed=True):
    band_items[str(band)] = {}
    for domain in domains:
        cols = map_table.loc[map_table['domain'] == domain, 'item']
        share = group[cols].notna().mean()
        band_items[str(band)][domain] = list(share[share >= 0.5].index)

for band, groups in band_items.items():
    print(band, {domain: len(items) for domain, items in groups.items()})

[3, 6) {'cognitive': 39, 'motor': 80, 'language': 24, 'social_emotional': 23}
[6, 9) {'cognitive': 39, 'motor': 80, 'language': 24, 'social_emotional': 23}
[9, 18) {'cognitive': 52, 'motor': 91, 'language': 36, 'social_emotional': 22}
[21, 24) {'cognitive': 3, 'motor': 6, 'language': 4, 'social_emotional': 16}
[24, 27) {'cognitive': 3, 'motor': 6, 'language': 4, 'social_emotional': 16}
[27, 30) {'cognitive': 3, 'motor': 6, 'language': 4, 'social_emotional': 16}


In [44]:
supported = []
for band, row in coverage.iterrows():
    supported.append({
        'age_band': str(band),
        'age_low': int(band.left),
        'age_high': int(band.right),
        'min_age_seen': round(row['min_age'], 2),
        'max_age_seen': round(row['max_age'], 2),
        'rows': int(row['rows']),
        'complete_share': round(row['complete'], 2),
    })

json.dump({'bands': supported, 'items': band_items}, open('supported_bands.json', 'w'), indent=1)

In [45]:
# Model Training
# Load the train and test sets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
norms = pd.read_json('norms.json')

domains = ['cognitive', 'motor', 'language', 'social_emotional']
z_cols = ['z_' + d for d in domains]

print('train:', train.shape)
print('test:', test.shape)
print(norms.shape)

train: (2028, 15)
test: (533, 15)
(24, 6)


In [46]:
print(train.columns.tolist())
print(train['complete'].value_counts())
print(test['complete'].value_counts())

['subjid', 'agedays', 'age_months', 'age_band', 'sex', 'family', 'cognitive', 'motor', 'language', 'social_emotional', 'z_cognitive', 'z_motor', 'z_language', 'z_social_emotional', 'complete']
complete
True     1965
False      63
Name: count, dtype: int64
complete
True     516
False     17
Name: count, dtype: int64


In [47]:
print('children in both sets:', len(set(train['subjid']) & set(test['subjid'])))
print(train[z_cols].isna().sum())
print(norms.head(8))

children in both sets: 0
z_cognitive           63
z_motor                0
z_language             4
z_social_emotional     0
dtype: int64
   age_low  age_high            domain    n      mean       std
0        3         6         cognitive  174  0.741085  0.153665
1        3         6             motor  174  0.718678  0.156066
2        3         6          language  174  0.657088  0.117800
3        3         6  social_emotional  174  0.845577  0.119415
4        6         9         cognitive  202  0.816070  0.116298
5        6         9             motor  202  0.811696  0.120672
6        6         9          language  202  0.738449  0.121264
7        6         9  social_emotional  202  0.868489  0.130214


In [48]:
# Train the Isolation Forest
from sklearn.ensemble import IsolationForest

train_model = train[train['complete']].copy()
X_train = train_model[z_cols]
print(X_train.shape)

model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_train)

(1965, 4)


IsolationForest(contamination=0.05, random_state=42)

In [49]:
train_model['if_score'] = model.decision_function(X_train)
train_model['if_flag'] = model.predict(X_train) == -1

print('flagged:', train_model['if_flag'].sum(), 'of', len(train_model))
print(train_model['if_score'].describe())

flagged: 99 of 1965
count    1965.000000
mean        0.131895
std         0.066440
min        -0.225349
25%         0.095459
50%         0.149571
75%         0.180573
max         0.216469
Name: if_score, dtype: float64


In [50]:
print(train_model.groupby('if_flag')[z_cols].mean().round(2))

         z_cognitive  z_motor  z_language  z_social_emotional
if_flag                                                      
False           0.08     0.08        0.11                0.09
True           -1.48    -1.47       -1.96               -1.72


In [51]:
print(train_model.groupby('age_band')['if_flag'].mean().round(3))
print(train_model.groupby('family')['if_flag'].mean().round(3))

age_band
[21, 24)    0.038
[24, 27)    0.047
[27, 30)    0.058
[3, 6)      0.075
[6, 9)      0.059
[9, 18)     0.050
Name: if_flag, dtype: float64
family
other       0.060
vineland    0.046
Name: if_flag, dtype: float64


In [52]:
# Build the rule-based baseline
norms['threshold'] = norms['mean'] - 2 * norms['std']
print(norms[['age_low', 'domain', 'threshold']].round(2).to_string())

    age_low            domain  threshold
0         3         cognitive       0.43
1         3             motor       0.41
2         3          language       0.42
3         3  social_emotional       0.61
4         6         cognitive       0.58
5         6             motor       0.57
6         6          language       0.50
7         6  social_emotional       0.61
8         9         cognitive       0.59
9         9             motor       0.66
10        9          language       0.72
11        9  social_emotional       0.66
12       21         cognitive       0.38
13       21             motor       0.50
14       21          language       0.45
15       21  social_emotional       0.56
16       24         cognitive       0.40
17       24             motor       0.48
18       24          language       0.56
19       24  social_emotional       0.62
20       27         cognitive       0.52
21       27             motor       0.55
22       27          language       0.75
23       27  soc

In [53]:
def rule_flag(data):
    data = data.copy()
    data['band_low'] = data['age_band'].str.extract(r'\[(\d+),')[0].astype(int)
    for domain in domains:
        table = norms[norms['domain'] == domain].set_index('age_low')['threshold']
        data['low_' + domain] = data[domain] < data['band_low'].map(table)
    data['rule_flag'] = data[['low_' + d for d in domains]].any(axis=1)
    return data

train_model = rule_flag(train_model)
train_all = rule_flag(train)
test_all = rule_flag(test)

In [54]:
print('flagged:', train_model['rule_flag'].sum(), 'of', len(train_model), round(train_model['rule_flag'].mean(), 3))
print(train_model[['low_' + d for d in domains]].sum())

flagged: 296 of 1965 0.151
low_cognitive           146
low_motor                59
low_language             94
low_social_emotional     90
dtype: int64


In [55]:
print(train_model.groupby('age_band')['rule_flag'].mean().round(3))
print(train_model.groupby('family')['rule_flag'].mean().round(3))

age_band
[21, 24)    0.165
[24, 27)    0.178
[27, 30)    0.161
[3, 6)      0.103
[6, 9)      0.089
[9, 18)     0.114
Name: rule_flag, dtype: float64
family
other       0.102
vineland    0.172
Name: rule_flag, dtype: float64


In [56]:
same = train_model['rule_flag'] == (train_model[z_cols] < -2).any(axis=1)
print('matches the z-score below -2 rule:', same.all())

matches the z-score below -2 rule: True


In [57]:
partial = train_all[~train_all['complete']]
print('rows with a missing domain:', len(partial), 'flagged:', partial['rule_flag'].sum())

rows with a missing domain: 63 flagged: 11


In [58]:
# Cross-validate with GroupKFold
from sklearn.model_selection import GroupKFold

train_model = train_model.reset_index(drop=True)
train_model['fold'] = -1
train_model['cv_score'] = np.nan
train_model['cv_flag'] = False

gkf = GroupKFold(n_splits=5)
fold_models = []

for fold, (fit_idx, val_idx) in enumerate(gkf.split(train_model, groups=train_model['subjid'])):
    fit_children = set(train_model.loc[fit_idx, 'subjid'])
    val_children = set(train_model.loc[val_idx, 'subjid'])
    print('fold', fold, 'children in both:', len(fit_children & val_children))

    fold_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    fold_model.fit(train_model.loc[fit_idx, z_cols])
    fold_models.append(fold_model)

    X_val = train_model.loc[val_idx, z_cols]
    train_model.loc[val_idx, 'fold'] = fold
    train_model.loc[val_idx, 'cv_score'] = fold_model.decision_function(X_val)
    train_model.loc[val_idx, 'cv_flag'] = fold_model.predict(X_val) == -1

fold 0 children in both: 0
fold 1 children in both: 0
fold 2 children in both: 0
fold 3 children in both: 0
fold 4 children in both: 0


In [59]:
cv_table = train_model.groupby('fold').agg(
    rows=('subjid', 'size'),
    children=('subjid', 'nunique'),
    if_flag_rate=('cv_flag', 'mean'),
    rule_flag_rate=('rule_flag', 'mean'),
    mean_score=('cv_score', 'mean'),
    std_score=('cv_score', 'std'),
)
print(cv_table.round(3))
print('mean IF flag rate:', round(cv_table['if_flag_rate'].mean(), 3))
print('std of IF flag rate:', round(cv_table['if_flag_rate'].std(), 3))

      rows  children  if_flag_rate  rule_flag_rate  mean_score  std_score
fold                                                                     
0      393       316         0.053           0.165       0.124      0.067
1      393       316         0.053           0.173       0.128      0.068
2      393       317         0.064           0.155       0.131      0.070
3      393       317         0.025           0.089       0.137      0.056
4      393       317         0.053           0.170       0.129      0.072
mean IF flag rate: 0.05
std of IF flag rate: 0.014


In [60]:
print(train_model.groupby('cv_flag')[z_cols].mean().round(2))
print('flagged in-sample (issue 11):', train_model['if_flag'].sum(), ' flagged out-of-fold:', train_model['cv_flag'].sum())

         z_cognitive  z_motor  z_language  z_social_emotional
cv_flag                                                      
False           0.07     0.07        0.10                0.09
True           -1.42    -1.41       -1.86               -1.73
flagged in-sample (issue 11): 99  flagged out-of-fold: 98


In [61]:
# Sensitivity check on contamination and n_estimators
from scipy.stats import spearmanr

folds = list(gkf.split(train_model, groups=train_model['subjid']))
rule = train_model['rule_flag'].values
seeds = [0, 1, 2, 3, 4]
pairs = [(i, j) for i in range(5) for j in range(i + 1, 5)]

def out_of_fold(n_estimators, contamination, seed):
    scores = np.zeros(len(train_model))
    flags = np.zeros(len(train_model), dtype=bool)
    for fit_idx, val_idx in folds:
        m = IsolationForest(n_estimators=n_estimators, contamination=contamination, random_state=seed)
        m.fit(train_model.loc[fit_idx, z_cols])
        X_val = train_model.loc[val_idx, z_cols]
        scores[val_idx] = m.decision_function(X_val)
        flags[val_idx] = m.predict(X_val) == -1
    return scores, flags

def jaccard(a, b):
    return (a & b).sum() / (a | b).sum()

In [62]:
rows = []
for contamination in [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
    runs = [out_of_fold(100, contamination, seed)[1] for seed in seeds]
    rows.append({
        'contamination': contamination,
        'flag_rate': np.mean([r.mean() for r in runs]),
        'fold_sd': np.mean([np.std([r[val].mean() for _, val in folds]) for r in runs]),
        'seed_agreement': np.mean([jaccard(runs[i], runs[j]) for i, j in pairs]),
        'also_rule_flagged': np.mean([(r & rule).sum() / r.sum() for r in runs]),
        'rule_flags_caught': np.mean([(r & rule).sum() / rule.sum() for r in runs]),
    })

print(pd.DataFrame(rows).round(3))

   contamination  flag_rate  fold_sd  seed_agreement  also_rule_flagged  \
0           0.01      0.011    0.004           0.779              1.000   
1           0.02      0.021    0.005           0.731              0.971   
2           0.03      0.032    0.008           0.743              0.962   
3           0.05      0.053    0.014           0.813              0.936   
4           0.08      0.083    0.020           0.798              0.919   
5           0.10      0.102    0.024           0.802              0.893   
6           0.15      0.152    0.029           0.794              0.781   

   rule_flags_caught  
0              0.073  
1              0.132  
2              0.207  
3              0.328  
4              0.504  
5              0.607  
6              0.786  


In [63]:
rows = []
for n_estimators in [25, 50, 100, 200, 500]:
    runs = [out_of_fold(n_estimators, 0.05, seed) for seed in seeds]
    rows.append({
        'n_estimators': n_estimators,
        'score_correlation': np.mean([spearmanr(runs[i][0], runs[j][0])[0] for i, j in pairs]),
        'seed_agreement': np.mean([jaccard(runs[i][1], runs[j][1]) for i, j in pairs]),
        'flag_rate_sd': np.std([r[1].mean() for r in runs]),
    })

print(pd.DataFrame(rows).round(3))

   n_estimators  score_correlation  seed_agreement  flag_rate_sd
0            25              0.924           0.625         0.003
1            50              0.956           0.747         0.001
2           100              0.978           0.813         0.002
3           200              0.988           0.877         0.001
4           500              0.994           0.925         0.001


In [64]:
model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
model.fit(X_train)

train_model['if_score'] = model.decision_function(X_train)
train_model['if_flag'] = model.predict(X_train) == -1
print('flagged:', train_model['if_flag'].sum(), 'of', len(train_model))

flagged: 99 of 1965


In [65]:
# Evaluate on the held-out test set
test_model = test_all[test_all['complete']].copy()
X_test = test_model[z_cols]
test_model['if_score'] = model.decision_function(X_test)
test_model['if_flag'] = model.predict(X_test) == -1

print('test rows:', len(test_model))
print('Isolation Forest flagged:', test_model['if_flag'].sum(), round(test_model['if_flag'].mean(), 3))
print('rule-based flagged:', test_model['rule_flag'].sum(), round(test_model['rule_flag'].mean(), 3))

test rows: 516
Isolation Forest flagged: 27 0.052
rule-based flagged: 71 0.138


In [66]:
from scipy.stats import ks_2samp

print(train_model['if_score'].describe().round(3))
print(test_model['if_score'].describe().round(3))
print(ks_2samp(train_model['if_score'], test_model['if_score']))

count    1965.000
mean        0.133
std         0.066
min        -0.226
25%         0.096
50%         0.149
75%         0.178
max         0.217
Name: if_score, dtype: float64
count    516.000
mean       0.134
std        0.067
min       -0.165
25%        0.100
50%        0.151
75%        0.184
max        0.217
Name: if_score, dtype: float64
KstestResult(statistic=np.float64(0.03671223149298775), pvalue=np.float64(0.6242990064050304), statistic_location=np.float64(0.18064273206229092), statistic_sign=np.int8(1))


In [67]:
train_model['lowest_z'] = train_model[z_cols].min(axis=1)
test_model['lowest_z'] = test_model[z_cols].min(axis=1)
print('train:', round(spearmanr(train_model['if_score'], train_model['lowest_z'])[0], 2))
print('test:', round(spearmanr(test_model['if_score'], test_model['lowest_z'])[0], 2))

train: 0.74
test: 0.78


In [68]:
print(pd.crosstab(test_model['rule_flag'], test_model['if_flag'], rownames=['rule'], colnames=['isolation forest']))

both = (test_model['rule_flag'] & test_model['if_flag']).sum()
either = (test_model['rule_flag'] | test_model['if_flag']).sum()
print('share of Isolation Forest flags also flagged by the rule:', round(both / test_model['if_flag'].sum(), 2))
print('share of rule flags also flagged by the Isolation Forest:', round(both / test_model['rule_flag'].sum(), 2))
print('overlap (Jaccard):', round(both / either, 2))

isolation forest  False  True 
rule                          
False               445      0
True                 44     27
share of Isolation Forest flags also flagged by the rule: 1.0
share of rule flags also flagged by the Isolation Forest: 0.38
overlap (Jaccard): 0.38


In [69]:
test_model['n_low'] = (test_model[z_cols] < -2).sum(axis=1)
print(pd.crosstab(test_model['n_low'], test_model['if_flag'], rownames=['domains below -2'], colnames=['isolation forest']))

isolation forest  False  True 
domains below -2              
0                   445      0
1                    44     10
2                     0     13
3                     0      3
4                     0      1


In [70]:
print(test_model.groupby('age_band')[['if_flag', 'rule_flag']].mean().round(3))
print(test_model.groupby('family')[['if_flag', 'rule_flag']].mean().round(3))

          if_flag  rule_flag
age_band                    
[21, 24)    0.049      0.108
[24, 27)    0.077      0.206
[27, 30)    0.018      0.107
[3, 6)      0.048      0.119
[6, 9)      0.030      0.060
[9, 18)     0.036      0.091
          if_flag  rule_flag
family                      
other       0.037      0.085
vineland    0.060      0.162


In [71]:
# Assess consistency across folds
def z_from(data, stats):
    z = pd.DataFrame(index=data.index)
    for domain in domains:
        mean = data['age_band'].map(stats[(domain, 'mean')])
        std = data['age_band'].map(stats[(domain, 'std')])
        z[domain] = (data[domain] - mean) / std
    return z

if_flags = []
rule_flags = []

for fit_idx, val_idx in folds:
    fit_children = train_model.loc[fit_idx, 'subjid']
    fit_all = train_all[train_all['subjid'].isin(fit_children)]
    stats = fit_all.groupby('age_band')[domains].agg(['mean', 'std'])

    z_fit = z_from(train_model.loc[fit_idx], stats)
    z_test = z_from(test_model, stats)

    fold_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    fold_model.fit(z_fit)

    if_flags.append(fold_model.predict(z_test) == -1)
    rule_flags.append((z_test < -2).any(axis=1).values)

In [72]:
def stability(flags, name):
    flags = np.array(flags)
    agreement = [jaccard(flags[i], flags[j]) for i, j in pairs]
    counts = flags.sum(axis=0)
    print(name)
    print('  flag rate per fold:', flags.mean(axis=1).round(3))
    print('  mean agreement between folds:', round(np.mean(agreement), 2), 'lowest:', round(min(agreement), 2))
    print('  flagged in any fold:', (counts > 0).sum())
    print('  flagged in all 5 folds:', (counts == 5).sum())
    print('  flagged in only 1 fold:', (counts == 1).sum())

stability(if_flags, 'Isolation Forest')
stability(rule_flags, 'Rule-based')

Isolation Forest
  flag rate per fold: [0.05  0.058 0.06  0.045 0.06 ]
  mean agreement between folds: 0.75 lowest: 0.64
  flagged in any fold: 37
  flagged in all 5 folds: 20
  flagged in only 1 fold: 5
Rule-based
  flag rate per fold: [0.143 0.141 0.143 0.141 0.147]
  mean agreement between folds: 0.93 lowest: 0.86
  flagged in any fold: 79
  flagged in all 5 folds: 68
  flagged in only 1 fold: 2


In [73]:
# Review the highest-scoring records
top = test_model.sort_values('if_score').head(20)
print(top[['age_months', 'family', 'if_score'] + domains + z_cols + ['n_low']].round(2).to_string())

     age_months    family  if_score  cognitive  motor  language  social_emotional  z_cognitive  z_motor  z_language  z_social_emotional  n_low
390        7.59     other     -0.16       0.49   0.38      0.42              0.22        -2.83    -3.62       -2.65               -5.00      4
10        25.40  vineland     -0.14       0.00   0.33      0.25              0.69        -3.81    -2.76       -3.84               -1.37      3
447       14.49     other     -0.11       0.58   0.63      0.72              0.27        -2.13    -2.29       -1.94               -5.65      3
191       24.41  vineland     -0.10       1.00   0.50      0.25              0.56         0.73    -1.88       -3.84               -2.49      2
195       25.30  vineland     -0.09       1.00   0.67      0.25              0.53         0.73    -0.99       -3.84               -2.75      2
525       25.59  vineland     -0.08       0.67   0.67      0.25              0.50        -0.79    -0.99       -3.84               -3.05      2

In [74]:
print(top['family'].value_counts())
print('top 20 with two or more domains below -2:', (top['n_low'] >= 2).sum())

family
vineland    16
other        4
Name: count, dtype: int64
top 20 with two or more domains below -2: 15


In [75]:
train_model['n_low'] = (train_model[z_cols] < -2).sum(axis=1)
top_train = train_model.sort_values('if_score').head(20)
print(top_train['n_low'].value_counts().sort_index())
print(top_train['family'].value_counts())
print('lowest z-score in the top 20:', round(top_train[z_cols].min().min(), 2))

n_low
2     4
3    12
4     4
Name: count, dtype: int64
family
other       12
vineland     8
Name: count, dtype: int64
lowest z-score in the top 20: -7.29


In [76]:
low_language = test_model[test_model['z_language'] < -3.5]
print(low_language[['age_band', 'language', 'z_language']].round(2).value_counts())

age_band  language  z_language
[24, 27)  0.25      -3.84         8
Name: count, dtype: int64


In [77]:
flagged = test_model[test_model['if_flag']]
print('flagged:', len(flagged))
print('with any z-score below -3:', (flagged[z_cols].min(axis=1) < -3).sum())
print('with all four z-scores below -1:', (flagged[z_cols] < -1).all(axis=1).sum())

flagged: 27
with any z-score below -3: 12
with all four z-scores below -1: 4


In [78]:
flagged_train = train_model[train_model['if_flag']]
print(flagged_train.groupby('subjid').size().value_counts())

1    94
3     1
2     1
Name: count, dtype: int64
